
# Word2Vec: Learning Distributed Word Representations

**Course:** Deep Learning (B.Sc. Computer Science)  
**Course website:** https://fum-cs.github.io/deep-learning

---

## Learning Objectives

After completing this notebook, students should be able to:

1. Explain why one-hot vectors are insufficient for representing words.
2. Describe the distributional hypothesis.
3. Understand Word2Vec as an encoder–decoder style neural architecture.
4. Explain the Skip-Gram and CBOW training objectives.
5. Generate training examples from raw text.
6. Train and evaluate Word2Vec models using Gensim.
7. Use spaCy word vectors for semantic similarity tasks.
8. Visualize and interpret learned embeddings.



# 1. From One-Hot Encoding to Word Embeddings

Traditional one-hot representations suffer from three major limitations:

- Very high dimensionality.
- Sparse representations.
- No notion of semantic similarity.

For example, the distance between **cat** and **dog** is identical to the distance between **cat** and **refrigerator**.

Word embeddings address this issue by learning dense vectors in which semantically related words tend to be close to one another.



# 2. The Distributional Hypothesis

The central idea behind Word2Vec is:

> *You shall know a word by the company it keeps.*  
> — J. R. Firth

Words appearing in similar contexts tend to have similar meanings.

Examples:

- "The **cat** chased the mouse."
- "The **dog** chased the ball."

Because *cat* and *dog* occur in related contexts, their vector representations become similar during training.



# 3. Connection to Encoder–Decoder Architectures

In Lecture 8 you studied **encoder–decoder models**.

Word2Vec can be viewed as a very simple encoder–decoder architecture.

## Encoder

Input:

- a one-hot word vector

Encoder:

- embedding matrix lookup

Output:

- dense latent representation (embedding)

## Decoder

Input:

- latent representation

Output:

- probability distribution over vocabulary

The latent layer plays exactly the role of a compressed representation.

```
one-hot word
      ↓
   Encoder
      ↓
 latent vector
      ↓
   Decoder
      ↓
context prediction
```

Unlike autoencoders, Word2Vec is **self-supervised**:

- input = word
- target = neighboring words

The learned latent representation is what we finally keep.



# 4. Skip-Gram and CBOW

## Skip-Gram

Given a center word, predict surrounding words.

Example sentence:

> Anne was beginning to get very tired

Using a window size of 2:

Center word:

```
beginning
```

Context words:

```
Anne, was, to, get
```

Training pairs:

```
(beginning → Anne)
(beginning → was)
(beginning → to)
(beginning → get)
```

---

## CBOW

The reverse task.

Input:

```
Anne, was, to, get
```

Target:

```
beginning
```

CBOW is generally faster, while Skip-Gram often performs better on smaller corpora and rare words.



# 5. Preparing Training Data

The most important concept for students is understanding how training examples are created.

The following code demonstrates the sliding-window mechanism used by Word2Vec.


In [1]:

sentence = "Anne was beginning to get very tired".split()

window_size = 2

for i, center_word in enumerate(sentence):
    start = max(0, i - window_size)
    end = min(len(sentence), i + window_size + 1)

    context = [
        sentence[j]
        for j in range(start, end)
        if j != i
    ]

    print(f"Center: {center_word:10s} -> Context: {context}")


Center: Anne       -> Context: ['was', 'beginning']
Center: was        -> Context: ['Anne', 'beginning', 'to']
Center: beginning  -> Context: ['Anne', 'was', 'to', 'get']
Center: to         -> Context: ['was', 'beginning', 'get', 'very']
Center: get        -> Context: ['beginning', 'to', 'very', 'tired']
Center: very       -> Context: ['to', 'get', 'tired']
Center: tired      -> Context: ['get', 'very']



# 6. Using *Anne of Green Gables* as the Corpus

The course repository contains:

```text
data/Anne_of_Green_Gables.txt
```

We will use it as our training corpus.


In [2]:

from pathlib import Path
import re

corpus_path = Path("data/Anne_of_Green_Gables.txt")

text = corpus_path.read_text(encoding="utf-8")

text = text.lower()
text = re.sub(r"[^a-z\s]", " ", text)

tokens = text.split()

print("Number of tokens:", len(tokens))
print(tokens[:50])


Number of tokens: 107178
['title', 'anne', 'of', 'green', 'gables', 'author', 'lucy', 'maud', 'montgomery', 'release', 'date', 'ebook', 'last', 'updated', 'october', 'language', 'english', 'anne', 'of', 'green', 'gables', 'by', 'lucy', 'maud', 'montgomery', 'table', 'of', 'contents', 'chapter', 'i', 'mrs', 'rachel', 'lynde', 'is', 'surprised', 'chapter', 'ii', 'matthew', 'cuthbert', 'is', 'surprised', 'chapter', 'iii', 'marilla', 'cuthbert', 'is', 'surprised', 'chapter', 'iv', 'morning']



# 7. Creating Skip-Gram Training Pairs

The following function creates the training pairs used by Word2Vec.


In [3]:

def generate_skipgram_pairs(tokens, window_size=2):

    pairs = []

    for i, center in enumerate(tokens):

        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            pairs.append((center, tokens[j]))

    return pairs


pairs = generate_skipgram_pairs(tokens[:100], window_size=2)

pairs[:20]


[('title', 'anne'),
 ('title', 'of'),
 ('anne', 'title'),
 ('anne', 'of'),
 ('anne', 'green'),
 ('of', 'title'),
 ('of', 'anne'),
 ('of', 'green'),
 ('of', 'gables'),
 ('green', 'anne'),
 ('green', 'of'),
 ('green', 'gables'),
 ('green', 'author'),
 ('gables', 'of'),
 ('gables', 'green'),
 ('gables', 'author'),
 ('gables', 'lucy'),
 ('author', 'green'),
 ('author', 'gables'),
 ('author', 'lucy')]


# 8. Training Word2Vec with Gensim

We will use Gensim's implementation instead of implementing Word2Vec from scratch.

This allows us to focus on understanding the model rather than low-level optimization details.


In [4]:

from gensim.models import Word2Vec

sentences = [tokens]

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,          # Skip-Gram
    epochs=10
)


c:\Users\m.amintoosi\.conda\envs\pth-gpu\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



# 9. Exploring Learned Embeddings


In [5]:

model.wv.most_similar("anne", topn=10)


[('surprised', 0.9965387582778931),
 ('concert', 0.9963647127151489),
 ('confession', 0.9953275322914124),
 ('invited', 0.9946743249893188),
 ('school', 0.9944130778312683),
 ('life', 0.9940248727798462),
 ('tea', 0.9940096735954285),
 ('lily', 0.9939918518066406),
 ('history', 0.993578314781189),
 ('interest', 0.9934440851211548)]

In [6]:

model.wv.similarity("anne", "marilla")


0.98570067


# 10. Word Analogies

One of the most famous properties of embeddings is vector arithmetic.

Although results depend strongly on corpus size, we can still experiment with analogies.


In [7]:

model.wv.most_similar(
    positive=["woman", "king"],
    negative=["man"],
    topn=10
)


KeyError: "Key 'king' not present in vocabulary"


# 11. Visualization with t-SNE


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

words = [
    w for w in model.wv.index_to_key[:50]
]

vectors = np.array([model.wv[w] for w in words])

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=10
)

coords = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 8))

for word, (x, y) in zip(words, coords):
    plt.scatter(x, y)
    plt.annotate(word, (x, y))

plt.title("Word Embeddings from Anne of Green Gables")
plt.show()



# 12. Using spaCy Word Vectors

spaCy provides high-quality pre-trained embeddings.

These vectors are not trained on *Anne of Green Gables*; they come from large external corpora.


In [ ]:

# !pip install spacy
# !python -m spacy download en_core_web_md

import spacy

nlp = spacy.load("en_core_web_md")


In [ ]:

doc1 = nlp("cat")
doc2 = nlp("dog")
doc3 = nlp("car")

print("cat vs dog :", doc1.similarity(doc2))
print("cat vs car :", doc1.similarity(doc3))


In [ ]:

for word in ["anne", "school", "friend"]:
    token = nlp.vocab[word]
    print(word, token.has_vector, token.vector.shape)



# 13. Comparing Gensim and spaCy

| Gensim Word2Vec | spaCy |
|---|---|
| Trains embeddings | Usually uses pre-trained embeddings |
| Useful for learning Word2Vec | Useful in NLP applications |
| Full control over corpus | Ready-to-use vectors |
| Educational focus | Production focus |



# Summary

In this notebook you learned:

- Why embeddings are better than one-hot vectors.
- The distributional hypothesis.
- Word2Vec as a simple encoder–decoder architecture.
- The difference between Skip-Gram and CBOW.
- How Word2Vec training examples are generated.
- How to train Word2Vec using Gensim.
- How to use pre-trained vectors in spaCy.
- How to visualize and interpret word embeddings.

The key takeaway is that Word2Vec learns a **latent representation of words**, analogous to the latent representations learned by autoencoders and encoder–decoder networks studied earlier in the course.
